# Graph-Q-SAT &middot; GAT-Q-SAT &middot; GTv2-Q-SAT
*Learning a SAT branching heuristic with reinforcement learning over Graph Neural Networks.*

End-to-end, reproducible pipeline (setup &rarr; data &rarr; training &rarr; evaluation &rarr; results) for the
graph-based models of the [NeuroSAT](https://github.com/dmeoli/NeuroSAT) project. The implementation lives in
the `.py` modules; this notebook only orchestrates the steps.

## 1. Introduction
A SAT solver (CDCL) repeatedly **picks a variable to branch on**; the quality of this *branching heuristic*
drives performance. We learn it with **Deep Q-Learning (DQN)** over a **Graph Neural Network** that reads the
CNF as a bipartite variable/clause graph. Three variants form a lineage:

- **Graph-Q-SAT** &mdash; encode&ndash;process&ndash;decode graph network (baseline).
- **GAT-Q-SAT** &mdash; adds edge-aware **graph attention** (GATConv) to the core.
- **GTv2-Q-SAT** &mdash; a **GATv2 Transformer** block (NeuroBack-inspired) successor.

Metric: **MRIR** = median (MiniSat iterations / model iterations); `>1` means fewer iterations than MiniSat.

## 2. Setup

### 2.1 GPU, Google Drive and repository

In [ ]:
!nvidia-smi -L || echo 'NO GPU - set Runtime > Change runtime type > GPU'
import torch; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

import os
# Drive holds the checkpoints so they survive disconnects (APPROVE the popup).
try:
    from google.colab import drive
    drive.mount('/content/gdrive', force_remount=True)
    CKPT_DIR = '/content/gdrive/MyDrive/neuroSAT_ckpts'
except Exception as e:
    print('Drive mount failed (%s) -> local checkpoints' % e)
    CKPT_DIR = '/content/neuroSAT_ckpts'
os.makedirs(CKPT_DIR, exist_ok=True); print('checkpoints ->', CKPT_DIR)

if not os.path.isdir('/content/neuroSAT/.git'):
    !git clone --recurse-submodules https://github.com/dmeoli/NeuroSAT /content/neuroSAT
%cd /content/neuroSAT
!git pull origin master 2>&1 | tail -1
!git submodule sync --recursive >/dev/null 2>&1; git submodule update --init --recursive --force 2>&1 | tail -2

### 2.2 Install the modern stack

In [ ]:
!pip -q install torch-geometric gymnasium PyYAML tensorboardX scipy matplotlib
!sudo apt-get -qq install -y swig zlib1g-dev >/dev/null
import sys; PYV = f'python{sys.version_info.major}.{sys.version_info.minor}'
!sudo apt-get -qq install -y {PYV}-dev >/dev/null
print('build prereqs ready for', PYV)

### 2.3 Build the native MiniSat gym environment

In [ ]:
import sys, os
PYV = f'python{sys.version_info.major}.{sys.version_info.minor}'
NPINC = __import__('numpy').get_include()
%cd /content/neuroSAT/GQSAT/minisat
!make python-wrap PYTHON={PYV} NUMPY_INC="{NPINC}" 2>&1 | tail -4
%cd /content/neuroSAT/GQSAT
import minisat  # registers sat-v0 and loads the native solver
print('GQSAT env built:', os.path.exists('minisat/minisat/gym/_GymSolver.so'))

## 3. Preprocessing

### 3.1 Datasets
SATLIB benchmarks live in the shared `data/` hub: **graph colouring** (`flat*`, structured) and
**uniform-random 3-SAT** (`uf*`/`uuf*`).

In [ ]:
%cd /content/neuroSAT
!ls data/graph-coloring | head

### 3.2 Train / validation / test split
Models are trained on `flat50-115` and evaluated on the other sizes (in-distribution generalisation).

In [ ]:
!pip -q install split-folders
!bash train_val_test_split.sh graph-coloring

### 3.3 MiniSat baseline metadata
MRIR compares against MiniSat, so each evaluation set needs a `METADATA` file (precomputed MiniSat iterations).

In [ ]:
%cd /content/neuroSAT/GQSAT
for d in ['train/flat50-115', 'val/flat50-115']:
    !python add_metadata.py --eval-problems-paths ../data/graph-coloring/{d}

## 4. Model
**State**: the CNF as a bipartite graph (variable & clause nodes; edge feature = literal polarity).
**Agent**: a DQN whose Q-function is an *encode&ndash;process&ndash;decode* graph network &mdash; an encoder, a
core that does *M* message-passing steps, and a decoder that emits a Q-value per variable; the action is the
`argmax`. **GAT-Q-SAT** replaces the uniform neighbour aggregation in the core with multi-head `GATConv`
attention; **GTv2-Q-SAT** uses a pre-norm `GATv2` Transformer block. (Full architecture & math: see the slides /
report.) Implementation: `gqsat/models.py`, training loop in `dqn.py`.

## 5. Training
One shell entry point reproduces the published configuration (decision cap 500, `penalty_size` 0.1,
`grad_clip` 0.1, &hellip;): `bash train.sh <variant> <train> <val> <logdir>`. Checkpoints go to Drive and the
script **auto-resumes** &mdash; just re-run the cell after a disconnect.

### 5.1 Train the three variants

In [ ]:
%cd /content/neuroSAT/GQSAT
TRAIN = '../data/graph-coloring/train/flat50-115'
VAL   = '../data/graph-coloring/val/flat50-115'
for variant in ['graphqsat', 'gatqsat', 'gtv2qsat']:
    print(f'\n##### training {variant} #####')
    !bash train.sh {variant} {TRAIN} {VAL} {CKPT_DIR}/gqsat_{variant}

### 5.2 Monitor training

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/gdrive/MyDrive/neuroSAT_ckpts

## 6. Evaluation

### 6.1 Reproduce the published results
Run the *released* checkpoints (shipped in `GQSAT/runs/`) on a test set &mdash; the MRIR matches the paper.

In [ ]:
%cd /content/neuroSAT/GQSAT
!bash evaluate.sh runs/Dec08_08-39-57_e63e47f25457 model_50000.chkp ../data/graph-coloring/flat30-60 | grep -E "Results|median"

### 6.2 Evaluate our trained models
Evaluate each freshly-trained variant's latest checkpoint.

In [ ]:
%cd /content/neuroSAT/GQSAT
import glob, re, os
for variant in ['graphqsat', 'gatqsat', 'gtv2qsat']:
    cks = glob.glob(f'{CKPT_DIR}/gqsat_{variant}/model_*.chkp')
    if not cks: print(variant, '-> no checkpoint yet'); continue
    ck = max(cks, key=lambda p: int(re.search(r'model_(\d+)', p).group(1)))
    print(f'\n##### {variant}: {os.path.basename(ck)} #####')
    !bash evaluate.sh {CKPT_DIR}/gqsat_{variant} {os.path.basename(ck)} ../data/graph-coloring/flat30-60 | grep -E "median"

## 7. Results

### 7.1 MRIR tables and figures
Regenerate the result tables and plots from the evaluation logs (`runs/*/*.tsv`).

In [ ]:
%cd /content/neuroSAT/GQSAT
!python aggregate_results.py && python make_plots.py && python paper_analysis.py

### 7.2 Cross-domain transfer
Evaluate the colouring-trained checkpoints, with no retraining, on unseen SATLIB domains.

In [ ]:
%cd /content/neuroSAT/GQSAT
!python transfer_study.py

### 7.3 Verdict &mdash; Graph vs GAT vs GTv2
Compare the three trained variants head-to-head (run after all reach step 50000).

In [ ]:
%cd /content/neuroSAT/GQSAT
import os, re, glob, sys, subprocess, statistics
TESTSETS = ['flat30-60', 'flat75-180', 'flat125-301', 'flat150-360', 'flat200-479']
MED = re.compile(r'median_relative_score:\s*([0-9.]+)')
def latest(v):
    cks = glob.glob(f'{CKPT_DIR}/gqsat_{v}/model_*.chkp')
    return (max(cks, key=lambda p: int(re.search(r'model_(\d+)', p).group(1))) if cks else None)
def ev(v, ck, ds):
    cmd = [sys.executable, 'evaluate.py', '--env-name', 'sat-v0', '--core-steps', '-1', '--eps-final', '0.0',
           '--no_restarts', '--test_time_max_decisions_allowed', '500', '--eval-problems-paths',
           f'../data/graph-coloring/{ds}', '--model-dir', f'{CKPT_DIR}/gqsat_{v}', '--model-checkpoint', os.path.basename(ck)]
    o = subprocess.run(cmd, capture_output=True, text=True).stdout
    m = [float(x) for x in MED.findall(o)]; return statistics.fmean(m) if m else float('nan')
res = {}
for v in ['graphqsat', 'gatqsat', 'gtv2qsat']:
    ck = latest(v)
    if ck: res[v] = {ds: ev(v, ck, ds) for ds in TESTSETS}; print(v, 'step', re.search(r'model_(\d+)', ck).group(1))
print(f"\n{'set':14s}" + ''.join(f'{v:>11s}' for v in res))
for ds in TESTSETS: print(f'{ds:14s}' + ''.join(f'{res[v][ds]:11.2f}' for v in res))
mean = {v: statistics.fmean(res[v].values()) for v in res}
print(f"{'MEAN':14s}" + ''.join(f'{mean[v]:11.2f}' for v in res))
if 'gatqsat' in mean and 'gtv2qsat' in mean:
    d = mean['gtv2qsat'] - mean['gatqsat']
    print(f"\nVERDICT: GTv2 {mean['gtv2qsat']:.2f} vs GAT {mean['gatqsat']:.2f} ({d:+.2f}) -> "
          f"GTv2 {'IMPROVES' if d > 0 else 'does NOT improve'} on GAT.")

## 8. Conclusions
- Graph attention (**GAT-Q-SAT**) helps on **structured** problems (graph colouring), with the advantage
  **growing with problem size**, and improves **cross-domain robustness**; it does not help on uniform-random SAT.
- **GTv2-Q-SAT** is kept only if section 7.3 shows it improves on GAT-Q-SAT.
- The fixed-size CNN of AlphaZeroSAT (companion notebook) cannot generalise across sizes &mdash; the motivation
  for these graph methods.